# 🧠 Ecko Networking System
**A personal networking prep tool for the socially anxious founder.**

Run each cell in order. Fill in your own data — the AI generates everything from *your* voice, not a template.

---
**需要準備：** [Groq API key](https://console.groq.com/keys)（免費）

In [ ]:
# @title ⚙️ Step 1：安裝（只需跑一次）
!git clone https://github.com/onlycaxxy/eco-networking 2>/dev/null || (cd eco-networking && git pull)
%cd /content/eco-networking
!pip install -q -r requirements.txt
print('✅ 安裝完成')

In [ ]:
# @title 🔑 Step 2：API Key 設定
# 先在左側邊欄點「🔑 Secrets」，新增一筆：
#   Name:  GROQ_API_KEY
#   Value: 你的 Groq key（gsk_...）
# 記得把「Notebook access」開關打開

from google.colab import userdata
import yaml
from pathlib import Path

try:
    groq_key = userdata.get('GROQ_API_KEY')
    print('✅ Groq API key 讀取成功')
except Exception:
    groq_key = ''
    print('⚠️  找不到 GROQ_API_KEY，請確認 Secrets 設定後重跑')

In [ ]:
# @title 👤 Step 3：你的基本資料
# 填完後按左上角 ▶ 執行

name    = 'Cathy' # @param {type:"string"}
product = 'Ecko'  # @param {type:"string"}
tagline = 'AI journal app，幫助焦慮的思考者找回聲音' # @param {type:"string"}

# 寫入 config.yaml
config = {
    'user': {'name': name, 'product': product, 'tagline': tagline},
    'claude': {
        'supplier': 'Groq',
        'base_url': 'https://api.groq.com/openai/v1',
        'api_key':  groq_key,
        'model':    'llama-3.3-70b-versatile',
    }
}
Path('config.yaml').write_text(yaml.dump(config, allow_unicode=True))
print(f'✅  {name} / {product} — 設定完成')

In [ ]:
# @title 🧠 Step 4：你的第二大腦
# 這裡的內容會直接注入 AI 的 system prompt
# 越真實、越具體，生成的內容就越像你

pitch = '你對產品最順口的一句話介紹' # @param {type:"string"}
intro = '你真的會說出口的自我介紹（不超過 75 字）' # @param {type:"string"}
insight = '你最近在想的一個問題，或你會自然聊起的話題' # @param {type:"string"}
anxiety_tip = '你實際用過有效的社交焦慮應對策略' # @param {type:"string"}

from ecko.db import NetworkingDB
db = NetworkingDB()
db.init()

# 清掉舊的，填入新的
import sqlite3
conn = sqlite3.connect('events.db')
conn.execute('DELETE FROM brain')
conn.commit()
conn.close()

db.add_brain('pitch',       '我的 pitch',    pitch)
db.add_brain('intro',       '自我介紹',       intro)
db.add_brain('insight',     '破冰話題',       insight)
db.add_brain('anxiety_tip', '焦慮應對策略',   anxiety_tip)

print('✅  第二大腦已更新')
print(f'   pitch      → {pitch[:40]}…')
print(f'   intro      → {intro[:40]}…')
print(f'   insight    → {insight[:40]}…')
print(f'   anxiety    → {anxiety_tip[:40]}…')

In [ ]:
# @title 📅 Step 5：新增活動

event_name     = '活動名稱' # @param {type:"string"}
event_datetime = '2026-06-13 19:00' # @param {type:"string"}
event_type     = 'meetup' # @param ["meetup", "coffee_chat"]
event_location = '台北' # @param {type:"string"}
event_notes    = '參加者輪廓、活動形式、你想達成的目標' # @param {type:"string"}

import sqlite3
conn = sqlite3.connect('events.db')
conn.execute('DELETE FROM events')
conn.commit()
conn.close()

event_id = db.add_event(
    name=event_name,
    datetime=event_datetime,
    type_=event_type,
    location=event_location,
    notes=event_notes,
)
print(f'✅  活動已建立（ID {event_id}）')
print(f'   {event_name}  ｜  {event_datetime}  ｜  {event_location}')

In [ ]:
# @title ✨ Step 6：生成準備包
from ecko.llm import PrepGenerator
from setup import load_config
from IPython.display import Markdown, display

config  = load_config()
event   = db.get_event(event_id)
brain   = {
    'pitch':       db.get_brain('pitch'),
    'intro':       db.get_brain('intro'),
    'insight':     db.get_brain('insight'),
    'anxiety_tip': db.get_brain('anxiety_tip'),
}

print('⏳ 生成中...')
gen    = PrepGenerator(config)
result = gen.generate(event, brain)

header = f'# 準備包：{event.name}\n> 📅 {event.datetime}　｜　📍 {event.location}\n\n---\n\n'
footer = '''
---
## 🚨 焦慮應急包（固定版）
- 提早 10 分鐘到，先站角落觀察
- 手機鬧鐘設 45 分鐘，到了就有正當理由離開
- 如果沉默超過 5 秒，問「你最近在忙什麼？」——永遠有效
- **退場許可：你不需要解釋為什麼要走**
'''
display(Markdown(header + result + footer))